# ACE-Step 1.5 — original instrumental soundtrack (AI Heritage Short)
Generates one instrumental track from the storyboard's music settings. All logic lives in
`ace_step_runner.py` (in the `heritage-short-inputs` dataset). **Settings:** Accelerator = GPU, Internet = On.
Fails loudly if the model cannot load. It never synthesises a stand-in.

In [ ]:
# 1. Install ACE-Step 1.5 from source (docs list `uv sync`; a plain `pip install -e .` is the Kaggle-friendly route).
!git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
!cd /kaggle/working/ACE-Step-1.5 && pip install -q -e .

In [ ]:
# 2. GPU check.
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU and restart."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,1), "GB")

In [ ]:
# 3. Locate the dataset.
import sys
from pathlib import Path
DATA = next((p for p in Path("/kaggle/input").glob("*") if (p/"acestep_batch.json").is_file()), None)
assert DATA, "Dataset not attached: add the `heritage-short-inputs` dataset to this notebook."
sys.path.insert(0, str(DATA))

In [ ]:
# 4. Generate (downloads checkpoints from Hugging Face on first run, ~10 GB).
#    On out-of-memory set THINKING = False (disables the LM planner).
import ace_step_runner as a
THINKING = True
rep = a.run_acestep(DATA/"acestep_batch.json", "/kaggle/working/outputs/music/soundtrack.wav",
                    project_root="/kaggle/working/ACE-Step-1.5", seed=42, thinking=THINKING, lm_backend="pt")
print(rep)

In [ ]:
# 5. Listen, then package. On your PC: python kaggle/bundle.py unpack soundtrack_bundle.zip
from IPython.display import Audio
import shutil
display(Audio("/kaggle/working/outputs/music/soundtrack.wav"))
shutil.make_archive("/kaggle/working/soundtrack_bundle", "zip", "/kaggle/working/outputs/music")